# 213 — Validate the Graph

End-to-end sanity checks after running 111 (Layer 1), 211 (Layer 2 ingest), and 212 (embeddings). Verifies:

1. Expected node and edge counts per layer
2. Referential integrity — no orphan Sections, Requirements, Chunks, etc.
3. Uniqueness constraints registered
4. Embedding coverage, dimension consistency, and vector index health
5. Every anomaly pattern in `ANOMALY_REGISTRY` returns results (smoke test)
6. A cross-layer bridge query (Layer 1 Company → Layer 2 Requirement)

Prints `PASS` / `FAIL` per check and a summary at the end. Embedding checks are skipped gracefully if notebook 212 has not been run yet.

## Imports and connection

In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv

REPO = Path.cwd().parent
sys.path.insert(0, str(REPO))
load_dotenv(REPO / '.env')

from src.graph.connection import Neo4jConnection
from src.mcp.schema import ANOMALY_REGISTRY

conn = Neo4jConnection().connect()
print('Connected to', conn._uri)

results = []

def check(name, passed, detail=''):
    status = 'PASS' if passed else 'FAIL'
    results.append((status, name, detail))
    print(f'{status}  {name}' + (f'  — {detail}' if detail else ''))

Connected to neo4j+s://692d7258.databases.neo4j.io


## 1. Node counts per layer

Expectations use loose lower bounds (not exact counts) so re-runs with different ICIJ seeds still pass.

In [2]:
EXPECTED = {
    # Layer 1 (ICIJ subset — loose lower bounds)
    'Person':       (1, None),
    'Company':      (50, None),
    'Intermediary': (1, None),
    'Address':      (1, None),
    'Jurisdiction': (1, None),
    # Layer 2 (MAS 626 parse)
    'Regulation':  (1, 1),
    'Section':     (10, None),
    'Requirement': (100, None),
    'Threshold':   (1, None),
    'Chunk':       (100, None),
}

counts = {
    row['label']: row['n']
    for row in conn.run_query('MATCH (n) RETURN labels(n)[0] AS label, count(*) AS n')
}

print('Label                Count   Expected')
print('-' * 42)
for label, (lo, hi) in EXPECTED.items():
    n = counts.get(label, 0)
    ok = n >= lo and (hi is None or n <= hi)
    bound = f'>={lo}' + (f', <={hi}' if hi is not None else '')
    check(f'Node count {label:<13}', ok, f'{n} ({bound})')

Label                Count   Expected
------------------------------------------
PASS  Node count Person         — 19 (>=1)
PASS  Node count Company        — 131 (>=50)
PASS  Node count Intermediary   — 7 (>=1)
PASS  Node count Address        — 28 (>=1)
PASS  Node count Jurisdiction   — 3 (>=1)
PASS  Node count Regulation     — 1 (>=1, <=1)
PASS  Node count Section        — 15 (>=10)
PASS  Node count Requirement    — 146 (>=100)
PASS  Node count Threshold      — 23 (>=1)
PASS  Node count Chunk          — 267 (>=100)


## 2. Edge counts per type

In [3]:
EXPECTED_EDGES = [
    # Layer 1
    'IS_OFFICER_OF', 'INTERMEDIARY_OF', 'REGISTERED_AT',
    'SHARES_ADDRESS_WITH', 'INCORPORATED_IN',
    # Layer 2
    'HAS_SECTION', 'NEXT_SECTION', 'HAS_REQUIREMENT',
    'DEFINES_THRESHOLD', 'HAS_CHUNK', 'NEXT_CHUNK',
]

edge_counts = {
    row['rel']: row['n']
    for row in conn.run_query('MATCH ()-[r]->() RETURN type(r) AS rel, count(*) AS n')
}

print('Edge type              Count')
print('-' * 32)
for rel in EXPECTED_EDGES:
    n = edge_counts.get(rel, 0)
    check(f'Edge count {rel:<20}', n > 0, f'{n}')

Edge type              Count
--------------------------------
PASS  Edge count IS_OFFICER_OF         — 147
PASS  Edge count INTERMEDIARY_OF       — 98
PASS  Edge count REGISTERED_AT         — 138
PASS  Edge count SHARES_ADDRESS_WITH   — 23
PASS  Edge count INCORPORATED_IN       — 131
PASS  Edge count HAS_SECTION           — 15
PASS  Edge count NEXT_SECTION          — 14
PASS  Edge count HAS_REQUIREMENT       — 146
PASS  Edge count DEFINES_THRESHOLD     — 23
PASS  Edge count HAS_CHUNK             — 267
PASS  Edge count NEXT_CHUNK            — 121


## 3. Referential integrity — no orphan nodes

In [4]:
ORPHAN_CHECKS = [
    ('Section w/o Regulation',   "MATCH (s:Section)     WHERE NOT (:Regulation)-[:HAS_SECTION]->(s)        RETURN count(*) AS n"),
    ('Requirement w/o Section',  "MATCH (r:Requirement) WHERE NOT (:Section)-[:HAS_REQUIREMENT]->(r)       RETURN count(*) AS n"),
    ('Threshold w/o Requirement', "MATCH (t:Threshold)   WHERE NOT (:Requirement)-[:DEFINES_THRESHOLD]->(t) RETURN count(*) AS n"),
    ('Chunk w/o Requirement',    "MATCH (c:Chunk)       WHERE NOT (:Requirement)-[:HAS_CHUNK]->(c)          RETURN count(*) AS n"),
    ('Company w/o Jurisdiction', "MATCH (c:Company)     WHERE c.jurisdiction <> '' AND NOT (c)-[:INCORPORATED_IN]->(:Jurisdiction) RETURN count(*) AS n"),
    ('Person w/o company link',  "MATCH (p:Person)      WHERE NOT (p)-[:IS_OFFICER_OF]->(:Company)          RETURN count(*) AS n"),
]

for name, cypher in ORPHAN_CHECKS:
    n = conn.run_query(cypher)[0]['n']
    check(f'No orphans — {name:<30}', n == 0, f'{n} orphan(s)')

PASS  No orphans — Section w/o Regulation          — 0 orphan(s)
PASS  No orphans — Requirement w/o Section         — 0 orphan(s)
PASS  No orphans — Threshold w/o Requirement       — 0 orphan(s)
PASS  No orphans — Chunk w/o Requirement           — 0 orphan(s)
PASS  No orphans — Company w/o Jurisdiction        — 0 orphan(s)
PASS  No orphans — Person w/o company link         — 0 orphan(s)


## 4. Uniqueness constraints

In [5]:
EXPECTED_CONSTRAINTS = {
    ('Person', 'node_id'),
    ('Company', 'node_id'),
    ('Intermediary', 'node_id'),
    ('Address', 'node_id'),
    ('Jurisdiction', 'jurisdiction_id'),
    ('Regulation', 'regulation_id'),
    ('Section', 'section_id'),
    ('Requirement', 'requirement_id'),
    ('Threshold', 'threshold_id'),
    ('Chunk', 'chunk_id'),
}

found = set()
rows = conn.run_query(
    'SHOW CONSTRAINTS YIELD labelsOrTypes, properties, type '
    'WHERE type CONTAINS "UNIQUE" RETURN labelsOrTypes, properties'
)
for row in rows:
    for lbl in row['labelsOrTypes']:
        for prop in row['properties']:
            found.add((lbl, prop))

missing = EXPECTED_CONSTRAINTS - found
check('All uniqueness constraints present', not missing,
      f'missing {missing}' if missing else f'{len(found)} constraints found')

PASS  All uniqueness constraints present  — 10 constraints found


## 5. Embeddings and vector index

Runs after notebook 212. Checks:

- **5a** Every `Chunk` node has an `embedding` property
- **5b** All embeddings share the same dimension (no partial-reruns with mixed models)
- **5c** The `chunk_embeddings` vector index exists and is `ONLINE`
- **5d** Self-similarity smoke test — using any existing chunk vector to query the index returns at least one hit

If notebook 212 has not been run yet, the block is skipped and nothing fails.

In [7]:
# 5a. Coverage: every Chunk should carry an embedding
rows = conn.run_query(
    'MATCH (c:Chunk) RETURN count(c) AS total, count(c.embedding) AS with_emb'
)
total, with_emb = rows[0]['total'], rows[0]['with_emb']

if with_emb == 0:
    print('SKIP  No chunks have embeddings yet. Run 212_embed_chunks.ipynb first.')
else:
    check('Embedding coverage — all chunks embedded', with_emb == total, f'{with_emb}/{total}')

    # 5b. Dimension consistency — all embeddings same length
    rows = conn.run_query('''
        MATCH (c:Chunk) WHERE c.embedding IS NOT NULL
        RETURN size(c.embedding) AS dim, count(*) AS n
        ORDER BY n DESC
    ''')
    dims = {r['dim']: r['n'] for r in rows}
    check('Embedding dimension is consistent', len(dims) == 1, f'dims={dims}')
    dim = next(iter(dims))

    # 5c. Vector index exists and is ONLINE
    rows = conn.run_query(
        "SHOW INDEXES YIELD name, state, type WHERE name = 'chunk_embeddings' RETURN state, type"
    )
    if not rows:
        check("Vector index 'chunk_embeddings' exists", False, 'not found')
    else:
        state = rows[0]['state']
        idx_type = rows[0].get('type', '')
        check(
            "Vector index 'chunk_embeddings' ONLINE",
            state == 'ONLINE' and 'VECTOR' in (idx_type or '').upper(),
            f'state={state}, type={idx_type}',
        )

        # 5d. Self-similarity smoke test — use an existing chunk's vector to query the index
        rows = conn.run_query('''
            MATCH (c:Chunk) WHERE c.embedding IS NOT NULL
            WITH c LIMIT 1
            CALL db.index.vector.queryNodes('chunk_embeddings', 3, c.embedding)
            YIELD node, score
            RETURN count(node) AS n, max(score) AS max_score
        ''')
        n = rows[0]['n']
        max_score = rows[0]['max_score']
        detail = f'{n} hits, max_score={max_score:.3f}' if max_score is not None else f'{n} hits'
        check('Vector search returns results', n >= 1, detail)

PASS  Embedding coverage — all chunks embedded  — 267/267
PASS  Embedding dimension is consistent  — dims={1024: 267}
PASS  Vector index 'chunk_embeddings' ONLINE  — state=ONLINE, type=VECTOR
PASS  Vector search returns results  — 3 hits, max_score=1.000


## 6. Anomaly pattern smoke tests

Runs every Cypher in `ANOMALY_REGISTRY` and confirms each returns at least one row. Zero rows = either the data is too thin for that pattern, or the Cypher needs adjustment.

In [8]:
for name, pat in ANOMALY_REGISTRY.items():
    try:
        rows = conn.run_query(pat.cypher, pat.params or {})
        check(f'Pattern {name:<35}', len(rows) > 0, f'{len(rows)} row(s)')
    except Exception as e:
        check(f'Pattern {name:<35}', False, f'ERROR: {type(e).__name__}: {e}')

PASS  Pattern common_controller_across_shells      — 3 row(s)
PASS  Pattern layered_ownership                    — 30 row(s)
PASS  Pattern high_risk_jurisdiction               — 30 row(s)
PASS  Pattern shared_address_cluster               — 7 row(s)
PASS  Pattern intermediary_shell_network           — 1 row(s)
PASS  Pattern bearer_obscured_ownership            — 20 row(s)


## 7. Cross-layer bridge

No explicit Layer-1 ↔ Layer-2 edges exist — the agent bridges them at query time by reasoning *'this Company is incorporated in a high-risk jurisdiction, therefore MAS 626 paragraph 4.1(b) applies.'*

This query simulates that bridge: take a Company in a high-risk jurisdiction and pull MAS 626 Requirements in paragraph 4.1 (risk assessment).

In [9]:
rows = conn.run_query('''
    MATCH (c:Company)-[:INCORPORATED_IN]->(j:Jurisdiction {aml_risk_rating: 'high'})
    WITH c, j LIMIT 1
    MATCH (r:Requirement {regulation_id: 'MAS-626'})
    WHERE r.paragraph STARTS WITH '4.1'
    RETURN c.name AS company, j.name AS jurisdiction,
           r.paragraph AS para, substring(r.text, 0, 120) AS text_snippet
    ORDER BY r.paragraph LIMIT 5
''')
check('Cross-layer bridge query returns results', len(rows) > 0, f'{len(rows)} row(s)')
for row in rows:
    print(f"  {row['company']:<45} ({row['jurisdiction']})  para {row['para']}: {row['text_snippet']}...")

PASS  Cross-layer bridge query returns results  — 1 row(s)
  BLAIRMORE HOLDINGS, INC.                      (Panama)  para 4.1: A bank shall take appropriate steps to identify, assess and understand, its money laundering and terrorism financing ris...


## Summary

In [10]:
pass_count = sum(1 for s, *_ in results if s == 'PASS')
fail_count = sum(1 for s, *_ in results if s == 'FAIL')

print(f'\n{pass_count} PASS  /  {fail_count} FAIL  (total {len(results)})')

if fail_count:
    print('\nFailures:')
    for status, name, detail in results:
        if status == 'FAIL':
            print(f'  - {name}  ({detail})')

conn.close()


41 PASS  /  0 FAIL  (total 41)
